# 02 - AI4I 2020 exploration

UCI predictive-maintenance data, training the failure-probability head of **Model A**.

**The trap in this dataset:** `TWF`, `HDF`, `PWF`, `OSF`, `RNF` are the individual
failure modes, and their OR reproduces the `machine_failure` target. Any model given
them scores ~1.0 and has learned nothing. `load_ai4i` therefore returns the features,
the target and the modes as three separate objects.


In [ ]:
import sys, warnings
warnings.filterwarnings("ignore")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from twinmind.utils.config import load_config
from twinmind.utils.paths import figures_dir
from twinmind.utils.seed import set_seed

set_seed(42)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
FIGURES = figures_dir()
DPI = 150

def save(fig, name):
    """Save a figure to reports/figures as a 150-dpi PNG."""
    path = FIGURES / f"{name}.png"
    fig.savefig(path, dpi=DPI, bbox_inches="tight")
    print(f"saved {path.relative_to(path.parents[2])}")
    return path

config = load_config("data")


In [ ]:
from twinmind.data.loaders import load_ai4i

cfg = config.ai4i
features, target, modes = load_ai4i()

print(f"features {features.shape}: {list(features.columns)}")
print(f"target   {target.shape}, name={target.name!r}")
print(f"modes    {modes.shape}: {list(modes.columns)}")
assert not set(features.columns) & set(cfg.failure_mode_columns)
print("\nconfirmed: no failure-mode column is in the feature frame")
features.describe().T

## Class balance

Severely imbalanced, which drives the choice of loss and metric.

In [ ]:
counts = target.value_counts().sort_index()
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].bar(["no failure", "failure"], counts.to_numpy(),
            color=["#4C72B0", "#C44E52"], edgecolor="white")
for i, v in enumerate(counts.to_numpy()):
    axes[0].text(i, v, f"{v:,}\n{v/len(target)*100:.2f}%", ha="center", va="bottom")
axes[0].set_title(f"Class balance - {target.mean()*100:.2f}% positive")
axes[0].set_ylabel("rows")

mode_counts = modes.sum().sort_values(ascending=False)
axes[1].bar(mode_counts.index.str.upper(), mode_counts.to_numpy(),
            color="#C44E52", edgecolor="white")
for i, v in enumerate(mode_counts.to_numpy()):
    axes[1].text(i, v, str(int(v)), ha="center", va="bottom")
axes[1].set_title("Failure count by mode")
axes[1].set_ylabel("occurrences")
fig.suptitle("AI4I 2020 class balance and failure modes", fontweight="bold")
fig.tight_layout()
save(fig, "02_ai4i_class_balance");

## Why the mode columns are quarantined

The OR of the five modes reproduces the target almost exactly. This cell measures
that agreement, so the separation in `load_ai4i` is justified by evidence.


In [ ]:
agreement = (modes.max(axis=1) == target).mean()
without_rnf = (modes.drop(columns=["rnf"]).max(axis=1) == target).mean()
print(f"OR(all five modes)      == target for {agreement*100:.2f}% of rows")
print(f"OR(modes without RNF)   == target for {without_rnf*100:.2f}% of rows")
print(f"\nRNF ('random failure') is the only real discrepancy: "
      f"{int(modes['rnf'].sum())} rows flagged, "
      f"{int(((modes['rnf']==1) & (target==0)).sum())} of which are NOT failures.")

fig, ax = plt.subplots(figsize=(6, 4))
pd.Series({"OR(all modes)": agreement, "OR(no RNF)": without_rnf}).plot.barh(
    ax=ax, color="#C44E52")
ax.set_xlim(0.95, 1.0); ax.set_xlabel("agreement with target")
ax.set_title("Failure modes reconstruct the target", fontweight="bold")
fig.tight_layout()
save(fig, "02_ai4i_leakage_evidence");

## Feature distributions, split by outcome

In [ ]:
numeric = list(cfg.feature_columns)
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, column in zip(axes.ravel(), numeric):
    for label, colour, name in [(0, "#4C72B0", "no failure"), (1, "#C44E52", "failure")]:
        ax.hist(features.loc[target == label, column], bins=40, alpha=0.6,
                density=True, color=colour, label=name)
    ax.set_title(column, fontsize=10); ax.legend(fontsize=8)
axes.ravel()[-1].axis("off")
fig.suptitle("Feature distributions by outcome (density-normalised)", fontweight="bold")
fig.tight_layout()
save(fig, "02_ai4i_feature_distributions");

## Correlation

Including the modes and target here **for analysis only** — it makes the leakage
visible rather than leaving it as an assertion.


In [ ]:
combined = pd.concat(
    [features[numeric], modes, target.rename("machine_failure")], axis=1
)
fig, ax = plt.subplots(figsize=(11, 8.5))
sns.heatmap(combined.corr(), annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            square=True, cbar_kws={"shrink": 0.7}, annot_kws={"size": 7}, ax=ax)
ax.set_title("Correlation - note the mode columns against the target",
             fontweight="bold")
fig.tight_layout()
save(fig, "02_ai4i_correlation_heatmap");

print("\nCorrelation with the target:")
print(combined.corr()["machine_failure"].drop("machine_failure")
      .sort_values(ascending=False).to_string())